# Avaliação de Data Quality — voebem.silver.empresas

Avaliação completa da qualidade dos dados e do aspecto de negócio da tabela **voebem.silver.empresas**, que armazena o cadastro unificado de empresas aéreas (nacionais e estrangeiras) provenientes da ANAC.

## 1. Visão Geral e Completude

Contagem total de registros, distribuição por origem do cadastro e verificação de valores nulos/vazios por coluna.

In [0]:
%sql
-- Contagem total e por origem_cadastro
SELECT
  COUNT(*) AS total_registros,
  SUM(CASE WHEN origem_cadastro = 'nacional' THEN 1 ELSE 0 END) AS registros_nacional,
  SUM(CASE WHEN origem_cadastro = 'estrangeira' THEN 1 ELSE 0 END) AS registros_estrangeira,
  SUM(CASE WHEN origem_cadastro IS NULL OR origem_cadastro = '' THEN 1 ELSE 0 END) AS registros_sem_origem
FROM voebem.silver.empresas;

In [0]:
%sql
-- Completude: contagem de nulos, vazios e placeholder ".." por coluna
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN icao IS NULL OR icao = '' THEN 1 ELSE 0 END) AS icao_nulo_ou_vazio,
  SUM(CASE WHEN icao = '..' THEN 1 ELSE 0 END) AS icao_placeholder,
  SUM(CASE WHEN sigla_iata IS NULL OR sigla_iata = '' THEN 1 ELSE 0 END) AS sigla_iata_nulo_ou_vazio,
  SUM(CASE WHEN sigla_iata = '..' THEN 1 ELSE 0 END) AS sigla_iata_placeholder,
  SUM(CASE WHEN razao_social IS NULL OR TRIM(razao_social) = '' THEN 1 ELSE 0 END) AS razao_social_nulo_ou_vazio,
  SUM(CASE WHEN servico IS NULL OR TRIM(servico) = '' THEN 1 ELSE 0 END) AS servico_nulo_ou_vazio,
  SUM(CASE WHEN cidade IS NULL OR TRIM(cidade) = '' THEN 1 ELSE 0 END) AS cidade_nulo_ou_vazio,
  SUM(CASE WHEN uf IS NULL OR TRIM(uf) = '' THEN 1 ELSE 0 END) AS uf_nulo_ou_vazio,
  SUM(CASE WHEN situacao IS NULL OR TRIM(situacao) = '' THEN 1 ELSE 0 END) AS situacao_nulo_ou_vazio,
  SUM(CASE WHEN origem_cadastro IS NULL OR TRIM(origem_cadastro) = '' THEN 1 ELSE 0 END) AS origem_cadastro_nulo_ou_vazio
FROM voebem.silver.empresas;

In [0]:
%sql
-- Completude em percentual por coluna (apenas colunas de negocio)
SELECT
  ROUND(100.0 * SUM(CASE WHEN icao IS NULL OR icao = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_icao_ausente,
  ROUND(100.0 * SUM(CASE WHEN sigla_iata IS NULL OR sigla_iata = '' OR sigla_iata = '..' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_sigla_iata_ausente,
  ROUND(100.0 * SUM(CASE WHEN razao_social IS NULL OR TRIM(razao_social) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_razao_social_ausente,
  ROUND(100.0 * SUM(CASE WHEN servico IS NULL OR TRIM(servico) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_servico_ausente,
  ROUND(100.0 * SUM(CASE WHEN cidade IS NULL OR TRIM(cidade) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_cidade_ausente,
  ROUND(100.0 * SUM(CASE WHEN uf IS NULL OR TRIM(uf) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_uf_ausente,
  ROUND(100.0 * SUM(CASE WHEN situacao IS NULL OR TRIM(situacao) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_situacao_ausente,
  ROUND(100.0 * SUM(CASE WHEN origem_cadastro IS NULL OR TRIM(origem_cadastro) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_origem_cadastro_ausente
FROM voebem.silver.empresas;

## 2. Unicidade e Duplicatas

Verificação de registros duplicados nas colunas que deveriam ser identificadores únicos: `icao`, `sigla_iata` e `razao_social`.

In [0]:
%sql
-- Duplicatas por icao (ignorando nulos e vazios)
SELECT icao, COUNT(*) AS qtd
FROM voebem.silver.empresas
WHERE icao IS NOT NULL AND icao != '' AND icao != '..'
GROUP BY icao
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

In [0]:
%sql
-- Duplicatas por sigla_iata (ignorando nulos, vazios e "..")
SELECT sigla_iata, COUNT(*) AS qtd
FROM voebem.silver.empresas
WHERE sigla_iata IS NOT NULL AND sigla_iata != '' AND sigla_iata != '..'
GROUP BY sigla_iata
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

In [0]:
%sql
-- Duplicatas por razao_social
SELECT razao_social, COUNT(*) AS qtd
FROM voebem.silver.empresas
WHERE razao_social IS NOT NULL AND TRIM(razao_social) != ''
GROUP BY razao_social
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

## 3. Consistência e Validade de Formatos

Validação de formatos esperados: `icao` deve ter 3 letras; `sigla_iata` deve ter 2 letras; `uf` deve ser uma sigla válida de estado brasileiro; `situacao` e `origem_cadastro` devem conter apenas valores esperados.

In [0]:
%sql
-- Validacao do formato do codigo ICAO (3 letras maiusculas)
SELECT
  SUM(CASE WHEN icao IS NOT NULL AND icao != '' AND icao != '..' AND NOT RLIKE(icao, '^[A-Z]{3}$') THEN 1 ELSE 0 END) AS icao_formato_invalido,
  SUM(CASE WHEN icao IS NOT NULL AND icao != '' AND icao != '..' AND RLIKE(icao, '^[A-Z]{3}$') THEN 1 ELSE 0 END) AS icao_formato_valido
FROM voebem.silver.empresas;

In [0]:
%sql
-- Validacao do formato da sigla IATA (2 letras maiusculas)
SELECT
  SUM(CASE WHEN sigla_iata IS NOT NULL AND sigla_iata != '' AND sigla_iata != '..' AND NOT RLIKE(sigla_iata, '^[A-Z]{2}$') THEN 1 ELSE 0 END) AS sigla_iata_formato_invalido,
  SUM(CASE WHEN sigla_iata IS NOT NULL AND sigla_iata != '' AND sigla_iata != '..' AND RLIKE(sigla_iata, '^[A-Z]{2}$') THEN 1 ELSE 0 END) AS sigla_iata_formato_valido
FROM voebem.silver.empresas;

In [0]:
%sql
-- Validacao de UF: listar valores distintos e verificar se sao siglas validas
SELECT uf, COUNT(*) AS qtd
FROM voebem.silver.empresas
GROUP BY uf
ORDER BY qtd DESC;

In [0]:
%sql
-- UFs invalidas (fora da lista de 27 unidades federativas)
SELECT uf, COUNT(*) AS qtd
FROM voebem.silver.empresas
WHERE uf IS NOT NULL
  AND TRIM(uf) != ''
  AND uf NOT IN ('AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG','PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO')
GROUP BY uf
ORDER BY qtd DESC;

In [0]:
%sql
-- Valores distintos de situacao
SELECT situacao, COUNT(*) AS qtd
FROM voebem.silver.empresas
GROUP BY situacao
ORDER BY qtd DESC;

In [0]:
%sql
-- Valores distintos de origem_cadastro
SELECT origem_cadastro, COUNT(*) AS qtd
FROM voebem.silver.empresas
GROUP BY origem_cadastro
ORDER BY qtd DESC;

## 4. Distribuição de Categóricos de Negócio

Análise das distribuições de `servico` e `situacao` para entender a composição do cadastro.

In [0]:
%sql
-- Top 20 tipos de servico por frequencia
SELECT servico, COUNT(*) AS qtd
FROM voebem.silver.empresas
GROUP BY servico
ORDER BY qtd DESC
LIMIT 20;

## 5. Regras de Negócio

Verificações específicas do domínio de empresas aéreas: empresas estrangeiras devem ter codigo ICAO; empresas ativas sem codigo ICAO/IATA; consistencia entre origem_cadastro e presenca de codigo ICAO.

In [0]:
%sql
-- Empresas estrangeiras sem codigo ICAO (regra de negocio: estrangeiras deveriam ter ICAO)
SELECT icao, sigla_iata, razao_social, servico, cidade, uf, situacao, origem_cadastro
FROM voebem.silver.empresas
WHERE origem_cadastro = 'estrangeira'
  AND (icao IS NULL OR icao = '' OR icao = '..')
ORDER BY razao_social;

In [0]:
%sql
-- Empresas nacionais com codigo ICAO preenchido (verificar se e esperado)
SELECT icao, sigla_iata, razao_social, servico, situacao, origem_cadastro
FROM voebem.silver.empresas
WHERE origem_cadastro = 'nacional'
  AND icao IS NOT NULL AND icao != '' AND icao != '..'
ORDER BY razao_social;

In [0]:
%sql
-- Empresas inativas (situacao != ATIVA) — ainda presentes na tabela
SELECT icao, sigla_iata, razao_social, servico, uf, situacao, origem_cadastro
FROM voebem.silver.empresas
WHERE situacao IS NULL OR UPPER(TRIM(situacao)) != 'ATIVA'
ORDER BY razao_social;

In [0]:
%sql
-- Empresas ativas sem codigo ICAO nem IATA (sem identificadores aereos)
SELECT icao, sigla_iata, razao_social, servico, uf, origem_cadastro
FROM voebem.silver.empresas
WHERE (icao IS NULL OR icao = '' OR icao = '..')
  AND (sigla_iata IS NULL OR sigla_iata = '' OR sigla_iata = '..')
  AND (UPPER(TRIM(situacao)) = 'ATIVA' OR situacao IS NULL)
ORDER BY razao_social
LIMIT 50;

In [0]:
%sql
-- Empresas com UF preenchido mas cidade vazia (inconsistencia geografica)
SELECT razao_social, cidade, uf, origem_cadastro
FROM voebem.silver.empresas
WHERE (cidade IS NULL OR TRIM(cidade) = '')
  AND uf IS NOT NULL AND TRIM(uf) != ''
ORDER BY razao_social;

In [0]:
%sql
-- Verificacao do placeholder ".." em sigla_iata: quantas empresas usam ".." vs NULL vs valor real
SELECT
  CASE
    WHEN sigla_iata IS NULL THEN 'NULL'
    WHEN sigla_iata = '' THEN 'vazio'
    WHEN sigla_iata = '..' THEN 'placeholder ".."'
    ELSE 'valor real'
  END AS tipo_sigla_iata,
  COUNT(*) AS qtd
FROM voebem.silver.empresas
GROUP BY CASE
    WHEN sigla_iata IS NULL THEN 'NULL'
    WHEN sigla_iata = '' THEN 'vazio'
    WHEN sigla_iata = '..' THEN 'placeholder ".."'
    ELSE 'valor real'
  END
ORDER BY qtd DESC;

In [0]:
%sql
-- Verificacao do placeholder ".." em icao
SELECT
  CASE
    WHEN icao IS NULL THEN 'NULL'
    WHEN icao = '' THEN 'vazio'
    WHEN icao = '..' THEN 'placeholder ".."'
    ELSE 'valor real'
  END AS tipo_icao,
  COUNT(*) AS qtd
FROM voebem.silver.empresas
GROUP BY CASE
    WHEN icao IS NULL THEN 'NULL'
    WHEN icao = '' THEN 'vazio'
    WHEN icao = '..' THEN 'placeholder ".."'
    ELSE 'valor real'
  END
ORDER BY qtd DESC;

## 6. Resumo e Recomendações

Consolidação dos achados e recomendacoes de melhoria.

In [0]:
%sql
-- Resumo consolidado dos checks de data quality
WITH stats AS (
  SELECT
    COUNT(*) AS total_registros,
    SUM(CASE WHEN icao IS NULL OR icao = '' OR icao = '..' THEN 1 ELSE 0 END) AS icao_ausente,
    SUM(CASE WHEN sigla_iata IS NULL OR sigla_iata = '' OR sigla_iata = '..' THEN 1 ELSE 0 END) AS sigla_iata_ausente,
    SUM(CASE WHEN razao_social IS NULL OR TRIM(razao_social) = '' THEN 1 ELSE 0 END) AS razao_social_ausente,
    SUM(CASE WHEN servico IS NULL OR TRIM(servico) = '' THEN 1 ELSE 0 END) AS servico_ausente,
    SUM(CASE WHEN cidade IS NULL OR TRIM(cidade) = '' THEN 1 ELSE 0 END) AS cidade_ausente,
    SUM(CASE WHEN uf IS NULL OR TRIM(uf) = '' THEN 1 ELSE 0 END) AS uf_ausente,
    SUM(CASE WHEN situacao IS NULL OR TRIM(situacao) = '' THEN 1 ELSE 0 END) AS situacao_ausente,
    SUM(CASE WHEN origem_cadastro = 'nacional' THEN 1 ELSE 0 END) AS qtd_nacional,
    SUM(CASE WHEN origem_cadastro = 'estrangeira' THEN 1 ELSE 0 END) AS qtd_estrangeira
  FROM voebem.silver.empresas
)
SELECT
  total_registros,
  qtd_nacional,
  qtd_estrangeira,
  icao_ausente,
  ROUND(100.0 * icao_ausente / total_registros, 2) AS pct_icao_ausente,
  sigla_iata_ausente,
  ROUND(100.0 * sigla_iata_ausente / total_registros, 2) AS pct_sigla_iata_ausente,
  razao_social_ausente,
  servico_ausente,
  cidade_ausente,
  uf_ausente,
  situacao_ausente
FROM stats;